# Lesson 6: Hands-On Research Task

**Duration:** 60 minutes

## Learning Objectives
- Apply all learned concepts to a real research problem
- Work independently with structured guidance
- Debug and troubleshoot on your own
- Document your approach and results

## Task: Multi-class Image Classification

You will build and train a model to classify images on a provided dataset or your own research data.

### Instructions:
1. **Choose a dataset** (provided or bring your own)
2. **Explore the data** - understand class distribution, image sizes, etc.
3. **Prepare the data** - create train/val/test splits
4. **Build a model** - use transfer learning or train from scratch
5. **Train and evaluate** - track metrics and debug issues
6. **Interpret results** - what works well? What could improve?

## Step 1: Load and Explore Your Dataset

Start by loading your dataset and understanding its structure:

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision import datasets, models
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Option 1: Use a public dataset (e.g., PlantVillage for crop disease classification)
# Option 2: Load from a custom directory

# Example: Using a standard dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Load CIFAR-10 as example (in practice, use your own data)
dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# TODO: Replace above with your dataset
# For custom data:
# dataset = datasets.ImageFolder(root='path/to/data', transform=transform)

print(f'Dataset size: {len(dataset)}')
print(f'Number of classes: {len(dataset.classes)}')
print(f'Classes: {dataset.classes}')

# Analyze class distribution
labels = np.array(dataset.targets)
unique, counts = np.unique(labels, return_counts=True)
print(f'\nClass distribution:')
for cls, count in zip(unique, counts):
    print(f'  Class {cls}: {count} samples')

## Step 2: Create Train/Validation/Test Splits

Split your data properly to avoid data leakage:

In [ ]:
from sklearn.model_selection import train_test_split

# Create train/val/test splits (e.g., 70/15/15)
indices = np.arange(len(dataset))
labels_arr = np.array(dataset.targets)

# Split 1: Train vs. (Val+Test)
train_idx, test_idx = train_test_split(
    indices, test_size=0.30, stratify=labels_arr, random_state=42
)

# Split 2: Val vs. Test
val_test_labels = labels_arr[test_idx]
val_idx_rel, test_idx_rel = train_test_split(
    np.arange(len(test_idx)), test_size=0.5, stratify=val_test_labels, random_state=42
)
val_idx = test_idx[val_idx_rel]
test_idx = test_idx[test_idx_rel]

print(f'Train size: {len(train_idx)}')
print(f'Val size: {len(val_idx)}')
print(f'Test size: {len(test_idx)}')

# Create data loaders
from torch.utils.data import Subset

train_subset = Subset(dataset, train_idx)
val_subset = Subset(dataset, val_idx)
test_subset = Subset(dataset, test_idx)

batch_size = 32
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

print(f'\nTrain batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

## Step 3: Build and Train Your Model

Choose your approach and implement it. Here's a template:

In [ ]:
["# Choose your approach:\n# Approach 1: Transfer Learning (recommended for limited data)\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nmodel = models.resnet18(pretrained=True)\nmodel.fc = nn.Linear(model.fc.in_features, 10)\nmodel = model.to(device)\n\n# Freeze backbone\nfor param in model.parameters():\n    param.requires_grad = False\nfor param in model.fc.parameters():\n    param.requires_grad = True\n\n# OR Approach 2: Fine-tuning\n# Uncomment to use fine-tuning instead\n# for param in model.parameters():\n#     param.requires_grad = True\n\ncriterion = nn.CrossEntropyLoss()\noptimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)\n\nprint(f'Model: ResNet-18')\nprint(f'Device: {device}')\nprint(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.2f}M')"]]

## Step 4: Training Loop with Validation

Train and validate your model:

In [ ]:
["# Train your model\nnum_epochs = 5\nbest_val_acc = 0\nhistory = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}\n\nfor epoch in range(num_epochs):\n    # Training phase\n    model.train()\n    train_loss = 0.0\n    train_correct = 0\n    train_total = 0\n    \n    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1} Train'):\n        images, labels = images.to(device), labels.to(device)\n        \n        outputs = model(images)\n        loss = criterion(outputs, labels)\n        \n        optimizer.zero_grad()\n        loss.backward()\n        optimizer.step()\n        \n        train_loss += loss.item()\n        _, predicted = torch.max(outputs, 1)\n        train_total += labels.size(0)\n        train_correct += (predicted == labels).sum().item()\n    \n    # Validation phase\n    model.eval()\n    val_loss = 0.0\n    val_correct = 0\n    val_total = 0\n    \n    with torch.no_grad():\n        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1} Val'):\n            images, labels = images.to(device), labels.to(device)\n            outputs = model(images)\n            loss = criterion(outputs, labels)\n            \n            val_loss += loss.item()\n            _, predicted = torch.max(outputs, 1)\n            val_total += labels.size(0)\n            val_correct += (predicted == labels).sum().item()\n    \n    # Compute metrics\n    avg_train_loss = train_loss / len(train_loader)\n    avg_train_acc = train_correct / train_total\n    avg_val_loss = val_loss / len(val_loader)\n    avg_val_acc = val_correct / val_total\n    \n    history['train_loss'].append(avg_train_loss)\n    history['train_acc'].append(avg_train_acc)\n    history['val_loss'].append(avg_val_loss)\n    history['val_acc'].append(avg_val_acc)\n    \n    print(f'Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.4f}')\n    print(f'          - Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_acc:.4f}')\n    \n    # Save best model\n    if avg_val_acc > best_val_acc:\n        best_val_acc = avg_val_acc\n        torch.save(model.state_dict(), 'best_model.pth')\n        print('Model saved!')\n\nprint(f'\\nBest validation accuracy: {best_val_acc:.4f}')"]]

## Step 5: Evaluate on Test Set and Visualize Results

Analyze your final model performance:

In [ ]:
["# Plot training history\nfig, axes = plt.subplots(1, 2, figsize=(12, 4))\n\naxes[0].plot(history['train_loss'], label='Train')\naxes[0].plot(history['val_loss'], label='Val')\naxes[0].set_xlabel('Epoch')\naxes[0].set_ylabel('Loss')\naxes[0].set_title('Training and Validation Loss')\naxes[0].legend()\naxes[0].grid(True)\n\naxes[1].plot(history['train_acc'], label='Train')\naxes[1].plot(history['val_acc'], label='Val')\naxes[1].set_xlabel('Epoch')\naxes[1].set_ylabel('Accuracy')\naxes[1].set_title('Training and Validation Accuracy')\naxes[1].legend()\naxes[1].grid(True)\n\nplt.tight_layout()\nplt.show()\n\n# Test on held-out test set\nmodel.eval()\ntest_correct = 0\ntest_total = 0\n\nwith torch.no_grad():\n    for images, labels in test_loader:\n        images, labels = images.to(device), labels.to(device)\n        outputs = model(images)\n        _, predicted = torch.max(outputs, 1)\n        test_total += labels.size(0)\n        test_correct += (predicted == labels).sum().item()\n\ntest_accuracy = test_correct / test_total\nprint(f'Final Test Accuracy: {test_accuracy:.4f}')\nprint(f'\\nSummary:')\nprint(f'  Best Val Acc: {best_val_acc:.4f}')\nprint(f'  Test Acc: {test_accuracy:.4f}')"]]